In [7]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import itertools
import seaborn as sns
import time
import pandas as pd
import cv2

In [8]:
class EyeBoundingBoxDataset(Dataset):
    def __init__(self, subject_ids, root_dir, transform=None, apply_preprocessing=True):
        self.root_dir = root_dir
        self.subject_ids = subject_ids
        self.transform = transform
        self.apply_preprocessing = apply_preprocessing  # Whether to apply Gamma + Stretch
        self.data = []

        # Precompute gamma LUT once to save time
        gamma = 0.8
        self.gamma_LUT = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)

        for subject_id in self.subject_ids:
            subject_dir = os.path.join(root_dir, 'openEDS', 'openEDS', subject_id)
            bbox_file = os.path.join(root_dir, 'bbox', 'bbox', f"{subject_id}.txt")
            with open(bbox_file, 'r') as f:
                bboxes = [list(map(float, line.strip().split())) for line in f.readlines()]
            for i in range(len(bboxes) - 1):
                img0_path = os.path.join(subject_dir, f"{i}.png")
                img1_path = os.path.join(subject_dir, f"{i+1}.png")
                self.data.append((img0_path, img1_path, bboxes[i+1]))

    def __len__(self):
        return len(self.data)

    def preprocess_image(self, img_path):
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Could not load image from: {img_path}")

        # 1) Apply Gamma Correction
        img = cv2.LUT(img, self.gamma_LUT)

        # 2) Apply Percentile Stretch via Vectorized LUT
        p_low, p_high = np.percentile(img, (1, 99))
        lut_indices = np.arange(256)
        stretch_LUT = np.clip((lut_indices - p_low) * (255.0 / max(p_high - p_low, 1)), 0, 255).astype(np.uint8)
        img = cv2.LUT(img, stretch_LUT)

        return img

    def load_image(self, img_path):
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Could not load image from: {img_path}")
        return img

    def __getitem__(self, idx):
        img0_path, img1_path, bbox = self.data[idx]

        if self.apply_preprocessing:
            img0 = self.preprocess_image(img0_path)
            img1 = self.preprocess_image(img1_path)
        else:
            img0 = self.load_image(img0_path)
            img1 = self.load_image(img1_path)

        # Convert numpy arrays (H, W) -> (1, H, W) torch.Tensor
        img0 = torch.from_numpy(img0).unsqueeze(0).float() / 255.0  # scale 0-1
        img1 = torch.from_numpy(img1).unsqueeze(0).float() / 255.0

        if self.transform:
            img0 = self.transform(img0)
            img1 = self.transform(img1)

        diff = img1 - img0
        input_tensor = torch.cat((img1, diff), dim=0)  # shape (2, H, W)

        target = torch.tensor(bbox, dtype=torch.float32)
        return input_tensor, target

In [9]:
class LightweightBBoxCNN(nn.Module):
    def __init__(self, hidden_size=64):  # now configurable
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, hidden_size),  # updated hidden size
            nn.ReLU(),
            nn.Linear(hidden_size, 4)  # 4 = xmin, xmax, ymin, ymax
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x

In [13]:
def train_model_by_subject(root_dir, train_subjects, val_subjects, model=None, num_epochs=20, batch_size=16, lr=1e-3):
    # New transform: work directly on torch Tensors (resize expects tensor input)
    transform = transforms.Compose([
        transforms.Resize((64, 64)),  # works on tensors if input is tensor
        # Normalize can be added here if you want later
    ])

    train_dataset = EyeBoundingBoxDataset(train_subjects, root_dir, transform=transform, apply_preprocessing=True)
    val_dataset = EyeBoundingBoxDataset(val_subjects, root_dir, transform=transform, apply_preprocessing=True)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    device = next(model.parameters()).device if model else torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if model is None:
        model = LightweightBBoxCNN().to(device)
    else:
        model.to(device)

    torch.autograd.set_detect_anomaly(True)

    criterion = nn.SmoothL1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses = [], []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)

        avg_train_loss = train_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                val_loss += criterion(outputs, targets).item() * inputs.size(0)

        avg_val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(avg_val_loss)

        if (epoch + 1) % 5 == 0:
            torch.save(model.state_dict(), f'checkpoints/pp_cp_e{epoch + 1}')
            print('Checkpoint saved...')

        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")

    return model, train_losses, val_losses

In [14]:
def split_subjects_and_save(root_dir, output_file='subject_split.txt', seed=42):
    random.seed(seed)
    subject_path = os.path.join(root_dir, 'openEDS', 'openEDS')
    all_subjects = sorted([d for d in os.listdir(subject_path) if d.startswith('S_') and os.path.isdir(os.path.join(subject_path, d))])

    random.shuffle(all_subjects)
    n_total = len(all_subjects)
    n_train = int(0.7 * n_total)
    n_val = int(0.2 * n_total)

    train_subjects = all_subjects[:n_train]
    val_subjects = all_subjects[n_train:n_train + n_val]
    test_subjects = all_subjects[n_train + n_val:]

    with open(output_file, 'w') as f:
        f.write("Training Subjects:\n")
        for s in train_subjects:
            f.write(f"{s}\n")
        f.write("\nValidation Subjects:\n")
        for s in val_subjects:
            f.write(f"{s}\n")
        f.write("\nTest Subjects:\n")
        for s in test_subjects:
            f.write(f"{s}\n")

    print(f"Subject split saved to {output_file}")
    return train_subjects, val_subjects, test_subjects


In [ ]:
# Parameters
BATCH_SIZE = 4
HIDDEN_SIZE = 64
LEARNING_RATE = 0.01
NUM_EPOCHS = 10  # or whatever you want
ROOT_DIR = r"C:\Users\omarh\OneDrive - Georgia Institute of Technology\openEDS2019"

# 1. Split subjects
train_subjects, val_subjects, test_subjects = split_subjects_and_save(ROOT_DIR)

# 2. Initialize model with optimized hidden layer size
model = LightweightBBoxCNN(hidden_size=HIDDEN_SIZE)

# 3. Train the model
model, train_losses, val_losses = train_model_by_subject(
    root_dir=ROOT_DIR,
    train_subjects=train_subjects,
    val_subjects=val_subjects,
    model=model,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE
)


Subject split saved to subject_split.txt
Epoch 1/10 - Train Loss: 25.1008 - Val Loss: 14.5429
Epoch 2/10 - Train Loss: 13.0658 - Val Loss: 13.1074
Epoch 3/10 - Train Loss: 10.5026 - Val Loss: 12.5270
Epoch 4/10 - Train Loss: 9.4869 - Val Loss: 11.7366
Checkpoint saved...
Epoch 5/10 - Train Loss: 9.0334 - Val Loss: 12.3607


In [ ]:
def compute_iou(boxA, boxB):
    xA1, xA2, yA1, yA2 = boxA
    xB1, xB2, yB1, yB2 = boxB

    x_left = max(xA1, xB1)
    y_top = max(yA1, yB1)
    x_right = min(xA2, xB2)
    y_bottom = min(yA2, yB2)

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    inter = (x_right - x_left) * (y_bottom - y_top)
    areaA = (xA2 - xA1) * (yA2 - yA1)
    areaB = (xB2 - xB1) * (yB2 - yB1)
    union = areaA + areaB - inter

    return inter / union if union > 0 else 0.0

In [ ]:
def evaluate_model_miou(model, test_subjects, root_dir, device='cpu'):
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])

    model.to(device)
    model.eval()

    all_ious = []
    total_time = 0.0
    total_frames = 0

    for subject_id in tqdm(test_subjects, desc="Evaluating on test subjects"):
        subject_dir = os.path.join(root_dir, 'openEDS', 'openEDS', subject_id)
        bbox_file = os.path.join(root_dir, 'bbox', 'bbox', f"{subject_id}.txt")

        if not os.path.exists(subject_dir) or not os.path.exists(bbox_file):
            continue

        with open(bbox_file, 'r') as f:
            gt_bboxes = [list(map(float, line.strip().split())) for line in f.readlines()]

        img_files = sorted([f for f in os.listdir(subject_dir) if f.endswith('.png')],
                           key=lambda x: int(x.replace('.png', '')))

        if len(img_files) != len(gt_bboxes):
            print(f"Skipping {subject_id} (mismatched frame count)")
            continue

        preds = []
        prev_img = None

        start = time.time()
        for idx, file in enumerate(img_files):
            img = Image.open(os.path.join(subject_dir, file)).convert('L')

            if idx == 0:
                preds.append([0, 0, 0, 0])
                prev_img = img
                continue

            img_t = transform(img).to(device)
            prev_img_t = transform(prev_img).to(device)
            diff = img_t - prev_img_t
            input_tensor = torch.cat((img_t, diff), dim=0).unsqueeze(0)

            with torch.no_grad():
                bbox = model(input_tensor).squeeze(0).cpu().numpy().tolist()

            # Clamp bbox to image bounds
            xmin = max(0, min(bbox[0], 640))
            xmax = max(0, min(bbox[1], 640))
            ymin = max(0, min(bbox[2], 400))
            ymax = max(0, min(bbox[3], 400))

            preds.append([xmin, xmax, ymin, ymax])
            prev_img = img

        end = time.time()

        ious = [compute_iou(gt, pred) for gt, pred in zip(gt_bboxes, preds)]
        subject_miou = np.mean(ious)
        all_ious.append(subject_miou)

        total_time += (end - start)
        total_frames += len(gt_bboxes)

    avg_time_per_frame = total_time / total_frames if total_frames > 0 else 0.0
    overall_miou = np.mean(all_ious) if all_ious else 0.0

    print(f"\n✅ Test mIoU across {len(all_ious)} subjects: {overall_miou:.4f}")
    print(f"⏱️ Average inference time per frame: {avg_time_per_frame:.4f} seconds")
    return overall_miou, avg_time_per_frame

In [ ]:
evaluate_model_miou(
    model=model,
    test_subjects=test_subjects,  # from your subject split
    root_dir=ROOT_DIR,
    device='cpu'
)

Evaluating on test subjects: 100%|██████████| 19/19 [00:18<00:00,  1.02it/s]


✅ Test mIoU across 19 subjects: 0.8549
⏱️ Average inference time per frame: 0.0066 seconds


(0.854917524820246, 0.006605617519122178)

In [ ]:
checkpoint_epochs = [10]
checkpoint_dir = "checkpoints"
root_dir = r"C:\Users\omarh\OneDrive - Georgia Institute of Technology\openEDS2019"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Evaluating all checkpoints...\n")
for epoch in checkpoint_epochs:
    checkpoint_path = os.path.join(checkpoint_dir, f"optim_cp_e{epoch}")
    model = LightweightBBoxCNN(hidden_size=64)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    miou, avg_time = evaluate_model_miou(model, test_subjects, root_dir, device=device)
    print(f"🧪 Epoch {epoch} → mIoU: {miou:.4f}, Avg Time/frame: {avg_time:.4f} sec")

C:\Users\omarh\AppData\Local\Temp\ipykernel_23888\2706506040.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path, map_locat

Evaluating all checkpoints...



Evaluating on test subjects: 100%|██████████| 19/19 [00:07<00:00,  2.60it/s]



✅ Test mIoU across 19 subjects: 0.8509
⏱️ Average inference time per frame: 0.0026 seconds
🧪 Epoch 10 → mIoU: 0.8509, Avg Time/frame: 0.0026 sec


Evaluating on test subjects: 100%|██████████| 19/19 [00:08<00:00,  2.37it/s]


...


✅ Test mIoU across 19 subjects: 0.8389
⏱️ Average inference time per frame: 0.0029 seconds
🧪 Epoch 30 → mIoU: 0.8389, Avg Time/frame: 0.0029 sec


Evaluating on test subjects: 100%|██████████| 19/19 [00:08<00:00,  2.33it/s]



✅ Test mIoU across 19 subjects: 0.8591
⏱️ Average inference time per frame: 0.0029 seconds
🧪 Epoch 40 → mIoU: 0.8591, Avg Time/frame: 0.0029 sec


Evaluating on test subjects: 100%|██████████| 19/19 [00:08<00:00,  2.32it/s]


✅ Test mIoU across 19 subjects: 0.8549
⏱️ Average inference time per frame: 0.0029 seconds
🧪 Epoch 50 → mIoU: 0.8549, Avg Time/frame: 0.0029 sec
